# LeetCode 1694 — Reformat Phone Number bằng Python

**Link đề:** https://leetcode.com/problems/reformat-phone-number/
**Độ khó:** Easy · **Chủ đề:** String

```python
class Solution:
    def reformatNumber(self, number: str) -> str:
        ...
```

## Mục tiêu học
1. Tách bài thành 2 pha rõ ràng: **làm sạch** (bỏ `' '` và `'-'`) → **chia khối** theo luật.
2. Nắm công cụ chuỗi Python: `str.isdigit()`, `str.replace()`, list comprehension, `"".join()`, slicing `s[i:j]`, `"-".join(blocks)`.
3. Viết lời giải con trỏ `i` với vòng `while`, rồi lời giải **công thức theo `n % 3`**.
4. Trace từng bước để thấy khi nào "cắt 3", khi nào "cắt 2 + 2".
5. Nhận diện pattern "chia nhóm cố định nhưng xử lý đuôi đặc biệt".

## 1. Đề bài và ví dụ

Cho chuỗi `number` gồm chữ số, dấu cách `' '` và/hoặc dấu gạch `'-'`. Định dạng lại:
1. Bỏ hết dấu cách và dấu gạch.
2. Nhóm chữ số từ trái sang phải thành các **khối 3** cho đến khi còn **≤ 4** chữ số.
3. Phần còn lại:
   - còn **2** chữ số → 1 khối dài 2;
   - còn **3** chữ số → 1 khối dài 3;
   - còn **4** chữ số → **2 khối dài 2**.
4. Nối các khối bằng `'-'`.

Không bao giờ có khối dài 1, và có tối đa hai khối dài 2.

| `number` | Chữ số | Kết quả |
|---|---|---|
| `"1-23-45 6"` | `123456` | `"123-456"` |
| `"123 4-567"` | `1234567` | `"123-45-67"` |
| `"123 4-5678"` | `12345678` | `"123-456-78"` |

**Constraints:** `2 <= len(number) <= 100`; chỉ gồm chữ số, `'-'`, `' '`; có **ít nhất 2 chữ số**.

### 1.1. Vì sao "còn 4 thì chia 2 + 2"?

Nếu còn 4 mà vẫn cắt 3 → dư 1 chữ số đứng một mình (`"123-4"`), vi phạm luật "không có khối dài 1". Vì vậy 4 phải tách thành `2 + 2`. Thử tính tay với 7 chữ số `1234567`: cắt `123`, còn `4567` (4 chữ số) → `45-67`.

In [1]:
digits = "1234567"
print("độ dài:", len(digits))
print("cắt 3 đầu:", digits[:3], "| còn lại:", digits[3:], "độ dài", len(digits[3:]))
rest = digits[3:]
print("còn 4 -> 2 + 2:", rest[:2], rest[2:])

độ dài: 7
cắt 3 đầu: 123 | còn lại: 4567 độ dài 4
còn 4 -> 2 + 2: 45 67


## 2. Nền tảng Python cần biết

### 2.1. `str.isdigit()` — kiểm tra ký tự là chữ số

In [2]:
for ch in ["5", " ", "-", "0"]:
    print(repr(ch), "isdigit ->", ch.isdigit())

'5' isdigit -> True
' ' isdigit -> False
'-' isdigit -> False
'0' isdigit -> True


### 2.2. List comprehension và `"".join()` — lọc rồi ghép lại

`[ch for ch in s if cond]` tạo list các ký tự thoả điều kiện. `"".join(list_ký_tự)` nối chúng thành chuỗi. Đây là cách chuẩn để **lọc chuỗi** trong Python.

In [3]:
number = "1-23-45 6"
kept = [ch for ch in number if ch.isdigit()]
print(kept)
print("".join(kept))

['1', '2', '3', '4', '5', '6']
123456


### 2.3. `str.replace(old, new)` — cách làm sạch khác

`replace` trả về **chuỗi mới** (chuỗi Python bất biến), thay mọi `old` bằng `new`. Gọi hai lần để bỏ cả `' '` và `'-'`.

In [4]:
number = "123 4-5678"
clean = number.replace(" ", "").replace("-", "")
print(repr(number), "->", repr(clean))
print("number vẫn giữ nguyên:", repr(number))

'123 4-5678' -> '12345678'
number vẫn giữ nguyên: '123 4-5678'


### 2.4. Slicing `s[i:j]`

`s[i:j]` lấy ký tự từ chỉ số `i` đến **trước** `j`. Nếu `j` vượt quá độ dài, Python tự cắt tới cuối (không lỗi). Slicing tạo chuỗi mới dài `j - i`.

In [5]:
s = "12345678"
print(s[0:3], s[3:6], s[6:9])     # s[6:9] chỉ còn "78", không lỗi
print(s[6:], s[-2:])              # bỏ j: tới cuối; chỉ số âm đếm từ cuối

123 456 78
78 78


### 2.5. `"-".join(blocks)` — nối khối bằng dấu gạch

`sep.join(list_chuỗi)` đặt `sep` **giữa** các phần tử (không thêm ở đầu/cuối). Vì vậy ta gom các khối vào list rồi join một lần — không phải lo dấu `-` thừa.

In [6]:
blocks = ["123", "45", "67"]
print("-".join(blocks))
print("-".join(["78"]))       # 1 phần tử -> không có dấu gạch

123-45-67
78


### 2.6. `divmod(a, b)` — thương và dư cùng lúc

`divmod(n, 3)` trả `(n // 3, n % 3)`. Hữu ích cho lời giải công thức ở phần sau.

In [7]:
for n in [6, 7, 8, 2, 4]:
    q, r = divmod(n, 3)
    print(f"n = {n}: n // 3 = {q}, n % 3 = {r}")

n = 6: n // 3 = 2, n % 3 = 0
n = 7: n // 3 = 2, n % 3 = 1
n = 8: n // 3 = 2, n % 3 = 2
n = 2: n // 3 = 0, n % 3 = 2
n = 4: n // 3 = 1, n % 3 = 1


## 3. Cách trực giác — cắt dần chuỗi bằng slicing

Làm đúng nguyên văn đề:
1. `digits` = chuỗi đã làm sạch.
2. Trong khi `len(digits) > 4`: lấy `digits[:3]` làm khối, gán `digits = digits[3:]`.
3. Còn lại ≤ 4: nếu đúng 4 → hai khối `digits[:2]`, `digits[2:]`; ngược lại (2 hoặc 3) → một khối `digits`.

In [8]:
def reformat_slice(number: str) -> str:
    digits = "".join(ch for ch in number if ch.isdigit())
    blocks = []
    while len(digits) > 4:
        blocks.append(digits[:3])
        digits = digits[3:]          # tạo chuỗi mới mỗi lần
    if len(digits) == 4:
        blocks.append(digits[:2])
        blocks.append(digits[2:])
    else:
        blocks.append(digits)        # còn 2 hoặc 3
    return "-".join(blocks)

print(reformat_slice("1-23-45 6"))    # 123-456
print(reformat_slice("123 4-567"))    # 123-45-67
print(reformat_slice("123 4-5678"))   # 123-456-78

123-456
123-45-67
123-456-78


### Chi phí và nút thắt

Gọi `n` là số chữ số (≤ 100). Mỗi lần `digits = digits[3:]` **sao chép** phần còn lại → tổng số ký tự sao chép ≈ `n + (n-3) + (n-6) + ...` = `O(n²)`. Với `n ≤ 100` hoàn toàn không sao, nhưng đây là thói quen nên tránh khi chuỗi lớn.

**Cải tiến:** không cắt chuỗi gốc; thay vào đó dùng **con trỏ `i`** đánh dấu vị trí bắt đầu khối kế tiếp. Số chữ số còn lại là `n - i`.

In [9]:
# Đếm số ký tự bị sao chép bởi digits = digits[3:]
def copy_cost(n):
    cost, length = 0, n
    while length > 4:
        length -= 3
        cost += length
    return cost

for n in [10, 100, 1000, 10000]:
    print(f"n = {n:>5}: ký tự sao chép ≈ {copy_cost(n):>9} (n²/6 ≈ {n * n // 6})")

n =    10: ký tự sao chép ≈        11 (n²/6 ≈ 16)
n =   100: ký tự sao chép ≈      1616 (n²/6 ≈ 1666)
n =  1000: ký tự sao chép ≈    166166 (n²/6 ≈ 166666)
n = 10000: ký tự sao chép ≈  16661666 (n²/6 ≈ 16666666)


## 4. Insight — chỉ cần biết "còn bao nhiêu chữ số"

Quyết định ở mỗi bước chỉ phụ thuộc **số chữ số còn lại** `rem = n - i`:

| `rem` | Hành động |
|---|---|
| `> 4` | cắt khối 3, `i += 3` |
| `4` | cắt 2 rồi 2, kết thúc |
| `3` | cắt 3, kết thúc |
| `2` | cắt 2, kết thúc |

`rem` không bao giờ bằng 1 hoặc 0 khi kết thúc: vì ta chỉ cắt 3 khi `rem ≥ 5`, nên sau khi cắt `rem ≥ 2`.

## 5. Lời giải chính — con trỏ `i` + vòng `while`

Mổ xẻ từng đoạn:

**Đoạn 1 — làm sạch:** `digits = "".join(ch for ch in number if ch.isdigit())`, `n = len(digits)`.

**Đoạn 2 — cắt khối 3 khi còn > 4:**
```python
i = 0
while n - i > 4:
    blocks.append(digits[i:i+3]); i += 3
```

**Đoạn 3 — xử lý đuôi:** `rem = n - i` (2, 3 hoặc 4).

In [10]:
digits = "12345678"
n = len(digits)
i = 0
blocks = []
while n - i > 4:
    blocks.append(digits[i:i + 3])
    i += 3
print("sau đoạn 2: blocks =", blocks, "| i =", i, "| rem =", n - i)

sau đoạn 2: blocks = ['123', '456'] | i = 6 | rem = 2


In [11]:
def reformat_pointer(number: str) -> str:
    digits = "".join(ch for ch in number if ch.isdigit())   # đoạn 1
    n = len(digits)
    blocks = []
    i = 0
    while n - i > 4:                                         # đoạn 2
        blocks.append(digits[i:i + 3])
        i += 3
    rem = n - i                                              # đoạn 3
    if rem == 4:
        blocks.append(digits[i:i + 2])
        blocks.append(digits[i + 2:i + 4])
    else:                                                    # rem là 2 hoặc 3
        blocks.append(digits[i:])
    return "-".join(blocks)

for s in ["1-23-45 6", "123 4-567", "123 4-5678", "12", "--1 2 3--"]:
    print(repr(s), "->", reformat_pointer(s))

'1-23-45 6' -> 123-456
'123 4-567' -> 123-45-67
'123 4-5678' -> 123-456-78
'12' -> 12
'--1 2 3--' -> 123


## 6. TRACE từng bước

Ví dụ `number = "123 4-567"` → `digits = "1234567"`, `n = 7`.

| Bước | i | rem = n - i | Điều kiện | Khối thêm | blocks |
|---:|---:|---:|---|---|---|
| 1 | 0 | 7 | `7 > 4` → cắt 3 | `123` | `[123]` |
| 2 | 3 | 4 | `4 > 4` sai → đuôi; rem = 4 | `45`, `67` | `[123, 45, 67]` |

Kết quả `"123-45-67"`.

Ví dụ `n = 8` (`12345678`): cắt `123` (rem 8), cắt `456` (rem 5), còn rem = 2 → `78`.

In [12]:
def trace_reformat(number: str) -> str:
    digits = "".join(ch for ch in number if ch.isdigit())
    n = len(digits)
    print(f"number = {number!r} -> digits = {digits!r}, n = {n}")
    blocks, i, step = [], 0, 0
    while n - i > 4:
        step += 1
        blocks.append(digits[i:i + 3])
        print(f"  bước {step}: i = {i}, rem = {n - i} > 4 -> cắt {digits[i:i+3]!r} -> blocks = {blocks}")
        i += 3
    rem = n - i
    if rem == 4:
        blocks += [digits[i:i + 2], digits[i + 2:]]
        print(f"  đuôi: i = {i}, rem = 4 -> chia 2 + 2 -> blocks = {blocks}")
    else:
        blocks.append(digits[i:])
        print(f"  đuôi: i = {i}, rem = {rem} -> 1 khối -> blocks = {blocks}")
    ans = "-".join(blocks)
    print("  kết quả:", ans)
    return ans

trace_reformat("123 4-567")
trace_reformat("123 4-5678")
trace_reformat("1 2 3 4 5 6 7 8 9 0")

number = '123 4-567' -> digits = '1234567', n = 7
  bước 1: i = 0, rem = 7 > 4 -> cắt '123' -> blocks = ['123']
  đuôi: i = 3, rem = 4 -> chia 2 + 2 -> blocks = ['123', '45', '67']
  kết quả: 123-45-67
number = '123 4-5678' -> digits = '12345678', n = 8
  bước 1: i = 0, rem = 8 > 4 -> cắt '123' -> blocks = ['123']
  bước 2: i = 3, rem = 5 > 4 -> cắt '456' -> blocks = ['123', '456']
  đuôi: i = 6, rem = 2 -> 1 khối -> blocks = ['123', '456', '78']
  kết quả: 123-456-78
number = '1 2 3 4 5 6 7 8 9 0' -> digits = '1234567890', n = 10
  bước 1: i = 0, rem = 10 > 4 -> cắt '123' -> blocks = ['123']
  bước 2: i = 3, rem = 7 > 4 -> cắt '456' -> blocks = ['123', '456']
  đuôi: i = 6, rem = 4 -> chia 2 + 2 -> blocks = ['123', '456', '78', '90']
  kết quả: 123-456-78-90


'123-456-78-90'

## 7. Lời giải khác

### 7.1. Công thức theo `n % 3` — biết trước kích thước mọi khối

Gọi `n` là số chữ số.
- `n % 3 == 0` → toàn khối 3.
- `n % 3 == 2` → các khối 3, cuối cùng là một khối 2.
- `n % 3 == 1` → các khối 3, cuối cùng là **hai khối 2** (vì `1 + 3 = 4 = 2 + 2`). Trường hợp này cần `n ≥ 4`; `n ≥ 2` theo đề và `n % 3 == 1` ⇒ `n ≥ 4` ✓.

Ta tính danh sách độ dài khối `sizes` rồi cắt tuần tự.

In [13]:
def block_sizes(n: int) -> list:
    r = n % 3
    if r == 0:
        return [3] * (n // 3)
    if r == 2:
        return [3] * (n // 3) + [2]
    return [3] * (n // 3 - 1) + [2, 2]      # r == 1

for n in [2, 3, 4, 5, 6, 7, 8, 10]:
    sz = block_sizes(n)
    print(f"n = {n:>2}: sizes = {sz}, tổng = {sum(sz)}")

n =  2: sizes = [2], tổng = 2
n =  3: sizes = [3], tổng = 3
n =  4: sizes = [2, 2], tổng = 4
n =  5: sizes = [3, 2], tổng = 5
n =  6: sizes = [3, 3], tổng = 6
n =  7: sizes = [3, 2, 2], tổng = 7
n =  8: sizes = [3, 3, 2], tổng = 8
n = 10: sizes = [3, 3, 2, 2], tổng = 10


In [14]:
def reformat_formula(number: str) -> str:
    digits = "".join(ch for ch in number if ch.isdigit())
    blocks, i = [], 0
    for size in block_sizes(len(digits)):
        blocks.append(digits[i:i + size])
        i += size
    return "-".join(blocks)

print(reformat_formula("123 4-567"), reformat_formula("123 4-5678"), reformat_formula("1-23-45 6"))

123-45-67 123-456-78 123-456


### 7.2. Đệ quy

`f(d)` = định dạng chuỗi chữ số `d`. Cơ sở: `len(d) <= 3` → `d`; `len(d) == 4` → `d[:2] + "-" + d[2:]`. Bước: `d[:3] + "-" + f(d[3:])`.

In [15]:
def fmt_digits(d: str) -> str:
    if len(d) <= 3:
        return d
    if len(d) == 4:
        return d[:2] + "-" + d[2:]
    return d[:3] + "-" + fmt_digits(d[3:])

def reformat_recursive(number: str) -> str:
    return fmt_digits(number.replace(" ", "").replace("-", ""))

print(reformat_recursive("123 4-567"), reformat_recursive("1-23-45 6"))

123-45-67 123-456


### 7.3. Chèn dấu gạch khi duyệt từng chữ số (một lượt, không cần list khối)

Biết `sizes`, ta tính trước **các vị trí cần chèn `-`** (cộng dồn độ dài), rồi duyệt chữ số một lần.

In [16]:
def reformat_one_pass(number: str) -> str:
    digits = [ch for ch in number if ch.isdigit()]
    cut_after = set()
    pos = 0
    for size in block_sizes(len(digits))[:-1]:   # không chèn sau khối cuối
        pos += size
        cut_after.add(pos)
    out = []
    for idx, ch in enumerate(digits):
        if idx in cut_after:
            out.append("-")
        out.append(ch)
    return "".join(out)

print(reformat_one_pass("123 4-567"), reformat_one_pass("123 4-5678"))

123-45-67 123-456-78


### 7.4. Bảng so sánh

| Cách | Ý tưởng | Time | Space | Ghi chú |
|---|---|---|---|---|
| Cắt dần slicing | `digits = digits[3:]` | `O(n²)` (sao chép) | `O(n)` | sát đề nhất |
| Con trỏ `i` | `rem = n - i` | `O(n)` | `O(n)` | khuyên dùng |
| Công thức `n % 3` | biết trước kích thước | `O(n)` | `O(n)` | đẹp, cần lập luận |
| Đệ quy | `d[:3] + "-" + f(d[3:])` | `O(n²)` | `O(n²)` tổng chuỗi tạm | dễ đọc |
| Một lượt chèn `-` | vị trí cắt cộng dồn | `O(n)` | `O(n)` | kiểu "stream" |

`n` = số ký tự của `number` (≤ 100). Output dài `O(n)` nên Space tối thiểu là `O(n)`.

## 8. Code nộp LeetCode

In [17]:
class Solution:
    def reformatNumber(self, number: str) -> str:
        digits = "".join(ch for ch in number if ch.isdigit())
        n = len(digits)
        blocks = []
        i = 0
        while n - i > 4:
            blocks.append(digits[i:i + 3])
            i += 3
        if n - i == 4:
            blocks.append(digits[i:i + 2])
            blocks.append(digits[i + 2:])
        else:
            blocks.append(digits[i:])
        return "-".join(blocks)

sol = Solution()
print(sol.reformatNumber("1-23-45 6"), sol.reformatNumber("123 4-567"), sol.reformatNumber("123 4-5678"))

123-456 123-45-67 123-456-78


Bản công thức (gọn):

In [18]:
class SolutionFormula:
    def reformatNumber(self, number: str) -> str:
        d = number.replace(" ", "").replace("-", "")
        n = len(d)
        q, r = divmod(n, 3)
        if r == 1:                       # đuôi 4 = 2 + 2
            q -= 1
        blocks = [d[3 * k:3 * k + 3] for k in range(q)]
        tail = d[3 * q:]
        if len(tail) == 4:
            blocks += [tail[:2], tail[2:]]
        elif tail:
            blocks.append(tail)
        return "-".join(blocks)

print(SolutionFormula().reformatNumber("123 4-567"), SolutionFormula().reformatNumber("12"))

123-45-67 12


## 9. Vì sao đúng?

**Invariant của vòng `while`:** trước mỗi lần lặp, `blocks` chứa đúng các khối 3 mà đề sẽ tạo ra từ `digits[0:i]`, và `i` là bội của 3.
- Đề nói "nhóm thành khối 3 *cho đến khi còn ≤ 4*" — đúng điều kiện `n - i > 4` của ta.
- Khi vòng lặp dừng, `rem = n - i ≤ 4`. Vì trước lần cắt cuối `rem ≥ 5` ⇒ sau cắt `rem ≥ 2`; còn nếu không cắt lần nào thì `rem = n ≥ 2` (đề bảo đảm). Vậy `rem ∈ {2, 3, 4}` — đúng ba trường hợp đề liệt kê, ta xử lý cả ba.
- `"-".join` đặt đúng một dấu gạch giữa hai khối liên tiếp.

**Công thức `n % 3`:** cắt khối 3 lặp lại làm `rem` giảm 3 mỗi lần, nên `rem` cuối có cùng số dư mod 3 với `n` và nằm trong `{2,3,4}`: dư 0 → 3, dư 2 → 2, dư 1 → 4 (= 2 + 2).

## 10. Độ phức tạp

Gọi `L = len(number)`, `n` = số chữ số (`n ≤ L ≤ 100`).
- Làm sạch: `O(L)`.
- Con trỏ / công thức: mỗi chữ số được sao chép vào đúng một khối ⇒ `O(n)`; `join` `O(n)`.
- **Tổng Time `O(L)`, Space `O(L)`** (chuỗi `digits`, list `blocks`, chuỗi kết quả).
- Cách cắt dần bằng `digits = digits[3:]` và đệ quy: `O(n²)` vì sao chép phần đuôi nhiều lần.

## 11. Bộ test

1. Ba ví dụ đề.
2. Biên: đúng 2 chữ số, 3, 4 (phải ra `"12-34"`), 5 (`"123-45"`).
3. Dễ sai: nhiều dấu cách/gạch liên tiếp, ở đầu/cuối; `n % 3 == 1` lớn (7, 10, 100).
4. Đối chiếu mọi cách trên chuỗi ngẫu nhiên; kiểm tra **tính chất** kết quả (không khối dài 1, tối đa 2 khối dài 2, bỏ `-` thì ra đúng `digits`).

In [19]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"PASS {name}: {actual!r}")

S = Solution()
check("đề 1", S.reformatNumber("1-23-45 6"), "123-456")
check("đề 2", S.reformatNumber("123 4-567"), "123-45-67")
check("đề 3", S.reformatNumber("123 4-5678"), "123-456-78")
check("2 chữ số", S.reformatNumber("12"), "12")
check("3 chữ số", S.reformatNumber("1 2-3"), "123")
check("4 chữ số", S.reformatNumber("1234"), "12-34")
check("5 chữ số", S.reformatNumber("12345"), "123-45")
check("ký tự rác đầu/cuối", S.reformatNumber("--  1 2 3 4 5 6 7 --"), "123-45-67")
check("10 chữ số", S.reformatNumber("1234567890"), "123-456-78-90")

PASS đề 1: '123-456'
PASS đề 2: '123-45-67'
PASS đề 3: '123-456-78'
PASS 2 chữ số: '12'
PASS 3 chữ số: '123'
PASS 4 chữ số: '12-34'
PASS 5 chữ số: '123-45'
PASS ký tự rác đầu/cuối: '123-45-67'
PASS 10 chữ số: '123-456-78-90'


In [20]:
import random

def valid_format(ans: str, digits: str) -> bool:
    parts = ans.split("-")
    lens = [len(p) for p in parts]
    return ("".join(parts) == digits and 1 not in lens
            and lens.count(2) <= 2 and all(l in (2, 3) for l in lens))

impls = [reformat_slice, reformat_pointer, reformat_formula, reformat_recursive,
         reformat_one_pass, Solution().reformatNumber, SolutionFormula().reformatNumber]
random.seed(7)
for _ in range(1000):
    nd = random.randint(2, 60)
    digits = "".join(random.choice("0123456789") for _ in range(nd))
    s = ""
    for ch in digits:
        s += random.choice(["", "", " ", "-", " -"]) + ch
    outs = {f(s) for f in impls}
    assert len(outs) == 1, (s, outs)
    assert valid_format(outs.pop(), digits)
print("Đối chiếu 7 cài đặt + kiểm tra tính chất trên 1000 chuỗi ngẫu nhiên: PASS")

Đối chiếu 7 cài đặt + kiểm tra tính chất trên 1000 chuỗi ngẫu nhiên: PASS


## 12. Lỗi người mới hay gặp

### Lỗi 1 — Luôn cắt 3, để lại khối dài 1

In [21]:
def wrong_always3(number):
    d = "".join(ch for ch in number if ch.isdigit())
    return "-".join(d[i:i + 3] for i in range(0, len(d), 3))   # SAI khi n % 3 == 1

print("sai:", wrong_always3("1234567"), "| đúng:", reformat_pointer("1234567"))

sai: 123-456-7 | đúng: 123-45-67


**Sửa:** xử lý riêng phần đuôi khi còn 4 chữ số (2 + 2).

### Lỗi 2 — Điều kiện vòng lặp `>= 4` thay vì `> 4`

Khi còn đúng 4 mà vẫn cắt 3 → dư 1.

In [22]:
def wrong_ge4(number):
    d = "".join(ch for ch in number if ch.isdigit())
    blocks, i = [], 0
    while len(d) - i >= 4:          # SAI: phải là > 4
        blocks.append(d[i:i + 3]); i += 3
    rem = len(d) - i
    if rem == 4:
        blocks += [d[i:i + 2], d[i + 2:]]
    elif rem > 0:
        blocks.append(d[i:])
    return "-".join(blocks)

print("sai:", wrong_ge4("1234"), "| đúng:", reformat_pointer("1234"))

sai: 123-4 | đúng: 12-34


**Sửa:** `while n - i > 4`.

### Lỗi 3 — Ghép chuỗi bằng `+ "-"` rồi để thừa dấu gạch ở cuối

In [23]:
def wrong_trailing_dash(number):
    d = "".join(ch for ch in number if ch.isdigit())
    out, i = "", 0
    while len(d) - i > 4:
        out += d[i:i + 3] + "-"; i += 3
    rem = len(d) - i
    if rem == 4:
        out += d[i:i + 2] + "-" + d[i + 2:] + "-"     # SAI: thừa '-'
    else:
        out += d[i:] + "-"
    return out

print("sai:", repr(wrong_trailing_dash("123456")), "| đúng:", repr(reformat_pointer("123456")))

sai: '123-456-' | đúng: '123-456'


**Sửa:** gom khối vào list và dùng `"-".join(blocks)`; hoặc `out.rstrip("-")` (kém rõ ràng hơn).

### Lỗi 4 — Chỉ bỏ dấu cách, quên dấu gạch (hoặc ngược lại)

`number.replace(" ", "")` vẫn còn `'-'` → khối chứa dấu gạch, độ dài sai. **Sửa:** lọc theo `isdigit()` để an toàn với mọi ký tự không phải số.

## 13. Mẹo nhận dạng pattern

- "Chia thành nhóm kích thước cố định, nhưng **nhóm cuối có luật riêng**" → vòng lặp với điều kiện `còn lại > ngưỡng`, rồi xử lý đuôi; hoặc tính `n % k` để biết trước.
- Cần **nối với dấu phân cách** → gom list rồi `sep.join(...)`, tránh lỗi dấu thừa.
- Làm sạch chuỗi → `"".join(ch for ch in s if điều_kiện)`.
- Bài tương tự: 482 License Key Formatting, 1592 Rearrange Spaces Between Words, 68 Text Justification (khó), 1556 Thousand Separator.

## 14. Biến thể

### Biến thể 1 — Dấu phân cách hàng nghìn (LeetCode 1556): nhóm 3 **từ phải sang trái**

In [24]:
def thousand_separator(n: int) -> str:
    s = str(n)
    blocks = []
    i = len(s)
    while i > 3:
        blocks.append(s[i - 3:i])
        i -= 3
    blocks.append(s[:i])
    return ".".join(reversed(blocks))

print(thousand_separator(987), thousand_separator(1234), thousand_separator(1234567890))

987 1.234 1.234.567.890


### Biến thể 2 — Khối kích thước `k` tổng quát, không khối nào ngắn hơn 2

Ý tưởng cùng kiểu: cắt `k` trong khi còn `> k + 1`; phần đuôi ≤ `k + 1` thì chia đôi nếu `> k`.

In [25]:
def group_k(digits: str, k: int) -> str:
    blocks, i, n = [], 0, len(digits)
    while n - i > k + 1:
        blocks.append(digits[i:i + k]); i += k
    rem = n - i
    if rem > k:                        # rem == k + 1 -> chia gần đều
        half = rem // 2
        blocks += [digits[i:i + half], digits[i + half:]]
    else:
        blocks.append(digits[i:])
    return "-".join(blocks)

print(group_k("1234567", 3), "|", group_k("123456789", 4), "|", group_k("12345", 4))

123-45-67 | 1234-56-789 | 12-345


## 15. Bài tập (tăng dần)

> Hàm bài tập chỉ có `pass`, **không được gọi** khi chạy notebook.

### Bài 1 — Dự đoán output
Không chạy code: `"0 0 0 0"` và `"9-9-9-9-9-9-9-9-9-9-9"` (11 số 9) cho kết quả gì?

In [26]:
def exercise_1():
    # TODO: trả về tuple (kết quả 1, kết quả 2) bạn tính tay
    pass

# assert exercise_1() == (Solution().reformatNumber("0 0 0 0"), Solution().reformatNumber("9-9-9-9-9-9-9-9-9-9-9"))

### Bài 2 — Viết hàm làm sạch không dùng `isdigit`
Chỉ dùng `replace` hoặc so sánh `ch not in " -"`.

In [27]:
def clean_digits(number: str) -> str:
    # TODO
    pass

# assert clean_digits("1-23-45 6") == "123456"

### Bài 3 — Điền chỗ trống phần đuôi

In [28]:
def tail_blocks(tail: str) -> list:
    # tail có độ dài 2, 3 hoặc 4
    # TODO: trả về list khối theo luật của đề
    pass

# assert tail_blocks("12") == ["12"] and tail_blocks("123") == ["123"] and tail_blocks("1234") == ["12", "34"]

### Bài 4 — Tự viết lại lời giải bằng công thức `n % 3`

In [29]:
def reformat_mod3(number: str) -> str:
    # TODO: tính kích thước khối từ n % 3 rồi cắt
    pass

# assert reformat_mod3("123 4-567") == "123-45-67"

### Bài 5 — Biến thể License Key (LeetCode 482, rút gọn)
Cho chuỗi `s` gồm chữ, số và `'-'`, và số `k`. Bỏ `'-'`, đổi chữ thành **in hoa**, nhóm **từ phải sang trái** thành khối `k` (khối đầu có thể ngắn hơn), nối bằng `'-'`.

In [30]:
def license_key(s: str, k: int) -> str:
    # TODO
    pass

# assert license_key("5F3Z-2e-9-w", 4) == "5F3Z-2E9W"
# assert license_key("2-5g-3-J", 2) == "2-5G-3J"

### Bài 6 — Mini challenge: đếm số khối dài 2
Không tạo chuỗi kết quả, chỉ từ `n` (số chữ số, `n ≥ 2`) trả về số khối dài 2.

In [31]:
def count_two_blocks(n: int) -> int:
    # TODO: O(1)
    pass

# assert [count_two_blocks(n) for n in [2, 3, 4, 5, 6, 7]] == [1, 0, 2, 1, 0, 2]

## 16. Gợi ý 3 mức

**Bài 1:** (1) Đếm số chữ số trước. (2) 4 chữ số → 2+2; 11 chữ số: 11 % 3 = 2. (3) `"00-00"`; `"999-999-999-99"`.

**Bài 2:** (1) Bỏ hai loại ký tự. (2) `replace(" ", "")` rồi `replace("-", "")`. (3) Hoặc `"".join(ch for ch in s if ch not in " -")`.

**Bài 3:** (1) Chỉ trường hợp 4 là đặc biệt. (2) `if len(tail) == 4`. (3) `[tail[:2], tail[2:]]` hoặc `[tail]`.

**Bài 4:** (1) Dùng `block_sizes`. (2) Duyệt kích thước, cắt `d[i:i+size]`. (3) `"-".join(...)`.

**Bài 5:** (1) Làm sạch + `upper()`. (2) Khối đầu dài `len % k` (nếu khác 0). (3) Cắt khối đầu, sau đó cắt đều `k`.

**Bài 6:** (1) Chỉ phụ thuộc `n % 3`. (2) dư 0 → 0, dư 2 → 1, dư 1 → 2. (3) `[0, 2, 1][n % 3]`.

## 17. Đáp án tham khảo

In [32]:
def ans_exercise_1():
    return ("00-00", "999-999-999-99")

def ans_clean_digits(number):
    return "".join(ch for ch in number if ch not in " -")

def ans_tail_blocks(tail):
    if len(tail) == 4:
        return [tail[:2], tail[2:]]
    return [tail]

def ans_reformat_mod3(number):
    d = ans_clean_digits(number)
    blocks, i = [], 0
    for size in block_sizes(len(d)):
        blocks.append(d[i:i + size])
        i += size
    return "-".join(blocks)

def ans_license_key(s, k):
    d = s.replace("-", "").upper()
    first = len(d) % k
    blocks = [d[:first]] if first else []
    for i in range(first, len(d), k):
        blocks.append(d[i:i + k])
    return "-".join(blocks)

def ans_count_two_blocks(n):
    return [0, 2, 1][n % 3]

assert ans_exercise_1() == (Solution().reformatNumber("0 0 0 0"), Solution().reformatNumber("9-9-9-9-9-9-9-9-9-9-9"))
assert ans_clean_digits("1-23-45 6") == "123456"
assert ans_tail_blocks("12") == ["12"] and ans_tail_blocks("123") == ["123"] and ans_tail_blocks("1234") == ["12", "34"]
assert ans_reformat_mod3("123 4-567") == "123-45-67"
assert ans_license_key("5F3Z-2e-9-w", 4) == "5F3Z-2E9W"
assert ans_license_key("2-5g-3-J", 2) == "2-5G-3J"
assert [ans_count_two_blocks(n) for n in [2, 3, 4, 5, 6, 7]] == [1, 0, 2, 1, 0, 2]
for n in range(2, 60):
    assert ans_count_two_blocks(n) == Solution().reformatNumber("7" * n).split("-").count("77")
print("Tất cả đáp án PASS!")

Tất cả đáp án PASS!


## 18. Template nhớ nhanh + checklist submit

```python
class Solution:
    def reformatNumber(self, number: str) -> str:
        d = "".join(ch for ch in number if ch.isdigit())   # 1) làm sạch
        blocks, i, n = [], 0, len(d)
        while n - i > 4:                                     # 2) khối 3 khi còn > 4
            blocks.append(d[i:i+3]); i += 3
        if n - i == 4:                                       # 3) đuôi 4 -> 2 + 2
            blocks += [d[i:i+2], d[i+2:]]
        else:                                                #    đuôi 2 hoặc 3
            blocks.append(d[i:])
        return "-".join(blocks)                              # 4) nối
```

**Khung tư duy:** làm sạch → cắt đều trong khi "còn nhiều" → xử lý đuôi theo bảng → `join`.

**Checklist trước khi submit**
- [ ] Đã bỏ **cả** `' '` và `'-'`?
- [ ] Điều kiện vòng lặp là `> 4` (không phải `>= 4`)?
- [ ] Còn đúng 4 chữ số → `2 + 2`?
- [ ] Không có dấu `-` thừa ở đầu/cuối (dùng `join`)?
- [ ] Đã thử `n = 2, 3, 4, 5` và `n % 3 == 1`?
- [ ] Tên method `reformatNumber`, trả về `str`?